In [1]:
import sqlalchemy as db
import pandas as pd

In [2]:
engine = db.create_engine('sqlite:///../raw/data.db') #loads db from raw

In [48]:
#loads all tables from db splitting events if they have data or don't
data_stocks = pd.read_sql('SELECT * FROM Stocks', con=engine) #Loads Stocks table
data_fed = pd.read_sql('SELECT * FROM Macro', con=engine) #loads macro data table
data_events = pd.read_sql('SELECT * FROM Events WHERE actual != "None" OR previous != "None"', con=engine) #load events with data
events_no_data = pd.read_sql('SELECT * FROM Events WHERE actual = "None" AND previous = "None"', con=engine) #load events without data

In [19]:
data_stocks.head()

,date,ticker,open,high,low,close,volume
0,1993-01-29 00:00:00.000000,SPY,24.258655,24.258655,24.137965,24.241413,1003200
1,1993-02-01 00:00:00.000000,SPY,24.258648,24.413820,24.258648,24.413820,480500
2,1993-02-02 00:00:00.000000,SPY,24.396588,24.482795,24.344863,24.465553,201300
3,1993-02-03 00:00:00.000000,SPY,24.500034,24.741414,24.482793,24.724173,529400
4,1993-02-04 00:00:00.000000,SPY,24.810374,24.879340,24.534512,24.827616,531500


In [20]:
data_stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 44603 entries, 0 to 44602
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    44603 non-null  str    
 1   ticker  44603 non-null  str    
 2   open    44603 non-null  float64
 3   high    44603 non-null  float64
 4   low     44603 non-null  float64
 5   close   44603 non-null  float64
 6   volume  44603 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 2.4 MB


In [56]:
data_stocks['ticker'].unique()

<StringArray>
['SPY', 'QQQ', '^VIX', 'DX-Y.NYB', 'GC=F']
Length: 5, dtype: str

##Creates the technical analisis here for each stock

##Transform one colum for each fed series

In [21]:
data_fed.head()

,date,serie,value
0,1954-07-01 00:00:00.000000,FEDFUNDS,0.80
1,1954-08-01 00:00:00.000000,FEDFUNDS,1.22
2,1954-09-01 00:00:00.000000,FEDFUNDS,1.07
3,1954-10-01 00:00:00.000000,FEDFUNDS,0.85
4,1954-11-01 00:00:00.000000,FEDFUNDS,0.83


In [22]:
data_fed.info()

<class 'pandas.DataFrame'>
RangeIndex: 99434 entries, 0 to 99433
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    99434 non-null  str    
 1   serie   99434 non-null  str    
 2   value   99434 non-null  float64
dtypes: float64(1), str(2)
memory usage: 2.3 MB


In [57]:
cols = data_fed['serie'].unique()
cols

<StringArray>
[    'FEDFUNDS',          'DFF',       'T10Y2Y',       'T10Y3M',
         'GS10',          'GS2', 'BAMLH0A0HYM2',   'BAMLC0A0CM',
         'NFCI',       'UNRATE',     'CPIAUCSL',        'T5YIE',
       'T10YIE',         'ICSA',        'WALCL',         'M2SL',
         'SOFR',      'TEDRATE']
Length: 18, dtype: str

In [87]:
#get all dates from original dataframe
data_fed['date'] = pd.to_datetime(data_fed['date'])
#sort all unique dates
all_dates = sorted(data_fed['date'].unique(), reverse=False)
#create the new dataframe and set all sorted dates as index
data_fed_transposed = pd.DataFrame(index=all_dates)

#creates one new colum for each serie and asing all the values into the correct date
for colum in cols:
    series_data = data_fed[data_fed['serie'] == colum].set_index('date')['value']
    data_fed_transposed[colum] = series_data

In [88]:
data_fed_transposed.tail()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,SOFR,TEDRATE
2026-02-18,NaN,3.64,0.62,0.39,NaN,NaN,2.86,0.78,NaN,NaN,NaN,2.43,2.29,NaN,6622382.0,NaN,3.73,NaN
2026-02-19,NaN,3.64,0.61,0.39,NaN,NaN,2.88,0.79,NaN,NaN,NaN,2.43,2.29,206000.0,NaN,NaN,3.67,NaN
2026-02-20,NaN,3.64,0.60,0.39,NaN,NaN,2.86,0.78,-0.56814,NaN,NaN,2.43,2.28,NaN,NaN,NaN,3.66,NaN
2026-02-23,NaN,NaN,0.60,0.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.40,2.26,NaN,NaN,NaN,NaN,NaN
2026-02-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6613395.0,NaN,NaN,NaN


In [94]:
#now we need to fill al Nans with the previous values
data_fed_transposed = data_fed_transposed.ffill(axis=0) #ffill gets the last valid value and fills the next nans
data_fed_transposed.tail()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,SOFR,TEDRATE
2026-02-18,3.64,3.64,0.62,0.39,4.21,3.54,2.86,0.78,-0.56857,4.3,326.588,2.43,2.29,229000.0,6622382.0,22411.0,3.73,0.09
2026-02-19,3.64,3.64,0.61,0.39,4.21,3.54,2.88,0.79,-0.56857,4.3,326.588,2.43,2.29,206000.0,6622382.0,22411.0,3.67,0.09
2026-02-20,3.64,3.64,0.60,0.39,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.43,2.28,206000.0,6622382.0,22411.0,3.66,0.09
2026-02-23,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6622382.0,22411.0,3.66,0.09
2026-02-25,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6613395.0,22411.0,3.66,0.09


In [95]:
data_fed_transposed.loc['2007-01-01']

FEDFUNDS             5.25000
DFF                  5.17000
T10Y2Y              -0.11000
T10Y3M              -0.31000
GS10                 4.76000
GS2                  4.88000
BAMLH0A0HYM2         2.89000
BAMLC0A0CM           0.91000
NFCI                -0.62684
UNRATE               4.50000
CPIAUCSL           203.10000
T5YIE                2.26000
T10YIE               2.30000
ICSA            323000.00000
WALCL           865010.00000
M2SL              7080.10000
SOFR                     NaN
TEDRATE              0.47000
Name: 2007-01-01 00:00:00, dtype: float64

In [97]:
data_fed_transposed['SOFR'].dropna()

2018-04-03    1.83
2018-04-04    1.74
2018-04-05    1.75
2018-04-06    1.75
2018-04-07    1.75
              ... 
2026-02-18    3.73
2026-02-19    3.67
2026-02-20    3.66
2026-02-23    3.66
2026-02-25    3.66
Name: SOFR, Length: 2883, dtype: float64

In [86]:
data_fed_transposed = data_fed_transposed.dropna(axis=0, how='any')
data_fed_transposed.head()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,SOFR,TEDRATE
2018-04-03,1.69,1.69,0.51,1.04,2.87,2.38,3.71,1.16,-0.50034,4.1,249.577,1.99,2.08,221000.0,4401222.0,13975.8,1.83,0.60
2018-04-04,1.69,1.69,0.51,1.08,2.87,2.38,3.71,1.16,-0.50034,4.1,249.577,2.04,2.08,221000.0,4392198.0,13975.8,1.74,0.64
2018-04-05,1.69,1.69,0.53,1.11,2.87,2.38,3.59,1.14,-0.50034,4.1,249.577,1.93,2.08,232000.0,4392198.0,13975.8,1.75,0.64
2018-04-06,1.69,1.69,0.50,1.04,2.87,2.38,3.64,1.14,-0.49876,4.1,249.577,2.01,2.07,232000.0,4392198.0,13975.8,1.75,0.64
2018-04-07,1.69,1.69,0.50,1.04,2.87,2.38,3.64,1.14,-0.49876,4.1,249.577,2.01,2.07,232000.0,4392198.0,13975.8,1.75,0.64


## Transform events into one colum for event if not data and one colum of each type of data for those who have

In [81]:
data_events.head()

,date,event,actual,previous
0,2026-01-28 00:00:00,US Federal Funds Rate,3.75%,3.75%
1,2025-12-10 00:00:00,US Federal Funds Rate,3.75%,4.00%
2,2025-10-29 00:00:00,US Federal Funds Rate,4.00%,4.25%
3,2025-09-17 00:00:00,US Federal Funds Rate,4.25%,4.50%
4,2025-07-30 00:00:00,US Federal Funds Rate,4.50%,4.50%


In [49]:
data_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 4489 entries, 0 to 4488
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      4489 non-null   str  
 1   event     4489 non-null   str  
 2   actual    4489 non-null   str  
 3   previous  4489 non-null   str  
dtypes: str(4)
memory usage: 140.4 KB


In [55]:
data_events['event'].unique()

<StringArray>
[         'US Federal Funds Rate',                'US Core CPI m/m',
                     'US CPI m/m',                     'US CPI y/y',
                     'US PPI m/m',    'US Core PCE Price Index m/m',
  'US Non-Farm Employment Change',           'US Unemployment Rate',
 'US Average Hourly Earnings m/m',             'US Advance GDP q/q',
              'US Prelim GDP q/q',               'US Final GDP q/q',
            'US Retail Sales m/m',       'US Core Retail Sales m/m',
       'US Personal Spending m/m',         'US Personal Income m/m',
       'US ISM Manufacturing PMI',            'US ISM Services PMI',
            'US Building Permits',              'US Housing Starts',
         'US Existing Home Sales',              'US New Home Sales']
Length: 22, dtype: str

In [47]:
events_no_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      305 non-null    str  
 1   event     305 non-null    str  
 2   actual    305 non-null    str  
 3   previous  305 non-null    str  
dtypes: str(4)
memory usage: 9.7 KB


In [53]:
events_no_data['event'].unique()

<StringArray>
['US FOMC Statement', 'US FOMC Press Conference',
 'US FOMC Economic Projections']
Length: 3, dtype: str